In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import sys

# srcディレクトリをPythonの検索パスに追加
sys.path.append("/home/keiseki/JR_train_snow/30.src")
from utils.utils import plot_feature_vs_target


warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
df_train = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/train_time_weather.pkl")
df_test = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/test_time_weather.pkl")

In [3]:
df_train_winter = df_train[df_train["月"].isin([12,1,2,3])]
df_test_winter = df_test[df_test["月"].isin([12,1,2,3])]

In [4]:
df_train_winter.to_pickle("train_winter_only.pkl")
df_test_winter.to_pickle("test_winter_only.pkl")

In [5]:
df_train_winter = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/train_winter_only.pkl")

In [6]:
df_train.shape

(15339, 1769)

In [7]:
df_train_winter.shape

(4603, 1769)

In [8]:
import pandas as pd
import numpy as np


def analyze_snow_rate_by_temperature(
    df,
    temp_col,
    target_col,
    temp_bins=None
):
    """
    気温帯ごとの着雪発生率を集計する。

    Parameters
    ----------
    df : pandas.DataFrame
        分析対象データ
    temp_col : str
        気温の列名
    target_col : str
        目的変数の列名
    temp_bins : list or None
        気温帯の境界。
        Noneの場合は1℃刻み。

    Returns
    -------
    summary : pandas.DataFrame
        気温帯ごとの件数・着雪発生件数・発生率など
    """

    data = df[[temp_col, target_col]].copy()

    # 数値化
    data[temp_col] = pd.to_numeric(
        data[temp_col],
        errors="coerce"
    )

    data[target_col] = pd.to_numeric(
        data[target_col],
        errors="coerce"
    )

    # 欠損除外
    data = data.dropna()

    # 気温帯を指定
    if temp_bins is None:
        temp_min = np.floor(data[temp_col].min())
        temp_max = np.ceil(data[temp_col].max())

        temp_bins = np.arange(
            temp_min,
            temp_max + 1,
            1
        )

    data["気温帯"] = pd.cut(
        data[temp_col],
        bins=temp_bins,
        right=False
    )

    # 着雪発生フラグ
    data["着雪発生"] = data[target_col] > 0

    # 集計
    summary = data.groupby(
        "気温帯",
        observed=True
    ).agg(
        件数=("着雪発生", "size"),
        着雪発生件数=("着雪発生", "sum"),
        平均着雪量=(target_col, "mean"),
        最大着雪量=(target_col, "max")
    )

    # 発生率
    summary["着雪発生率"] = (
        summary["着雪発生件数"]
        / summary["件数"]
        * 100
    )

    return summary

In [9]:
for col in df_train.columns:
    if "気温" in col:
        print(f"--- {col} ---")

        summary = analyze_snow_rate_by_temperature(
            df_train,
            temp_col=col,
            target_col="合計"
        )

        display(summary)

--- 富山_気温_℃__1_00 ---


,件数,着雪発生件数,平均着雪量,最大着雪量,着雪発生率
気温帯,,,,,
"[-4.0, -3.0)",50,25,0.007215,0.049420,50.000000
"[-2.0, -1.0)",178,6,0.000200,0.018895,3.370787
"[-1.0, 0.0)",354,5,0.000076,0.020128,1.412429
"[0.0, 1.0)",362,52,0.000775,0.023197,14.364641
"[1.0, 2.0)",575,18,0.000045,0.003791,3.130435
"[2.0, 3.0)",440,0,0.000000,0.000000,0.000000
"[3.0, 4.0)",572,0,0.000000,0.000000,0.000000
"[4.0, 5.0)",312,27,0.000187,0.015786,8.653846
"[5.0, 6.0)",264,0,0.000000,0.000000,0.000000


--- 富山_気温_℃__2_00 ---


,件数,着雪発生件数,平均着雪量,最大着雪量,着雪発生率
気温帯,,,,,
"[-4.0, -3.0)",50,25,0.007215,0.049420,50.000000
"[-2.0, -1.0)",178,5,0.000033,0.003999,2.808989
"[-1.0, 0.0)",354,6,0.000160,0.020128,1.694915
"[0.0, 1.0)",541,70,0.000567,0.023197,12.939002
"[1.0, 2.0)",352,0,0.000000,0.000000,0.000000
"[2.0, 3.0)",532,27,0.000110,0.015786,5.075188
"[3.0, 4.0)",660,0,0.000000,0.000000,0.000000
"[4.0, 5.0)",264,0,0.000000,0.000000,0.000000
"[5.0, 6.0)",264,0,0.000000,0.000000,0.000000


--- 富山_気温_℃__3_00 ---


...

,件数,着雪発生件数,平均着雪量,最大着雪量,着雪発生率
気温帯,,,,,
"[-4.0, -3.0)",50,25,0.007215,0.049420,50.000000
"[-1.0, 0.0)",132,0,0.000000,0.000000,0.000000
"[0.0, 1.0)",266,7,0.000210,0.020128,2.631579
"[1.0, 2.0)",362,54,0.000774,0.023197,14.917127
"[2.0, 3.0)",443,18,0.000059,0.003791,4.063205
"[3.0, 4.0)",662,2,0.000011,0.005348,0.302115
"[4.0, 5.0)",484,0,0.000000,0.000000,0.000000
"[5.0, 6.0)",352,0,0.000000,0.000000,0.000000
"[6.0, 7.0)",444,27,0.000132,0.015786,6.081081


--- 金沢_列車通過時_気温 ---


,件数,着雪発生件数,平均着雪量,最大着雪量,着雪発生率
気温帯,,,,,
"[-4.0, -3.0)",5,5,0.006110,0.014888,100.000000
"[-3.0, -2.0)",3,1,0.000842,0.002526,33.333333
"[-2.0, -1.0)",11,0,0.000000,0.000000,0.000000
"[-1.0, 0.0)",67,25,0.003872,0.037332,37.313433
"[0.0, 1.0)",180,36,0.000762,0.011760,20.000000
"[1.0, 2.0)",334,26,0.000504,0.049420,7.784431
"[2.0, 3.0)",359,13,0.000035,0.001921,3.621170
"[3.0, 4.0)",309,8,0.000046,0.005348,2.588997
"[4.0, 5.0)",340,1,0.000004,0.001195,0.294118


--- 新高岡_列車通過時_気温 ---


ValueError: arange: cannot compute length